<a href="https://colab.research.google.com/github/rananabil844/W2025/blob/main/Bachelor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json

kaggle_dict = {
    "username": "token",
    "key": "KGAT_6631afe2e38d3f3c138a6661f066fffd"
}

with open("kaggle.json", "w") as f:
    json.dump(kaggle_dict, f)

In [ ]:
!mkdir -p /root/.config/kaggle
!cp kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets download -d dhairyajeetsingh/ecommerce-customer-behavior-dataset
!unzip ecommerce-customer-behavior-dataset.zip

In [ ]:
import pandas as pd

df = pd.read_csv('ecommerce_customer_churn_dataset.csv')
df

## **Dataset Description**

The dataset consists of customer demographic, behavioral, transactional, and financial attributes collected from a digital commerce platform. It includes information related to purchasing activity, engagement levels, service interactions, marketing responsiveness, and tenure history. The dataset also contains a binary target variable indicating whether a customer has churned or remained active, enabling the development of a machine learning model for churn risk prediction.

In [ ]:
import pandas as pd

# Create Feature Description Table
feature_table = pd.DataFrame({

    "Feature_Name": [
        "Age",
        "Gender",
        "Country",
        "City",
        "Membership_Years",
        "Login_Frequency",
        "Session_Duration_Avg",
        "Pages_Per_Session",
        "Cart_Abandonment_Rate",
        "Wishlist_Items",
        "Total_Purchases",
        "Average_Order_Value",
        "Days_Since_Last_Purchase",
        "Discount_Usage_Rate",
        "Returns_Rate",
        "Email_Open_Rate",
        "Customer_Service_Calls",
        "Product_Reviews_Written",
        "Social_Media_Engagement_Score",
        "Mobile_App_Usage",
        "Payment_Method_Diversity",
        "Lifetime_Value",
        "Credit_Balance",
        "Signup_Quarter",
        "Churned"
    ],

    "Data_Type": [
        "Numeric",
        "Categorical",
        "Categorical",
        "Categorical",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Categorical",
        "Binary (0/1)"
    ],

    "Category": [
        "Demographic",
        "Demographic",
        "Demographic",
        "Demographic",
        "Customer_Tenure",
        "Behavioral",
        "Behavioral",
        "Behavioral",
        "Behavioral",
        "Engagement",
        "Transactional",
        "Transactional",
        "Behavioral (Recency)",
        "Pricing_Behavior",
        "Post_Purchase_Behavior",
        "Marketing_Engagement",
        "Customer_Support",
        "Engagement",
        "Digital_Engagement",
        "Digital_Behavior",
        "Financial_Behavior",
        "Financial",
        "Financial",
        "Acquisition",
        "Target"
    ],

    "Description": [
        "Customer age",
        "Customer gender",
        "Customer country",
        "Customer city",
        "Number of years as a registered customer",
        "Frequency of customer logins",
        "Average duration of sessions",
        "Average number of pages viewed per session",
        "Rate of abandoned shopping carts",
        "Number of items added to wishlist",
        "Total number of completed purchases",
        "Average monetary value per order",
        "Number of days since last purchase",
        "Rate of discount usage",
        "Rate of product returns",
        "Percentage of marketing emails opened",
        "Number of customer service interactions",
        "Number of product reviews written",
        "Level of social media interaction",
        "Frequency of mobile app usage",
        "Number of different payment methods used",
        "Total revenue generated by the customer",
        "Available credit balance",
        "Quarter of customer registration",
        "Churn status (1 = churned, 0 = active)"
    ],

})

feature_table

In [ ]:
df

In [ ]:
original_customer_45467_features = df.loc[45467]
display(original_customer_45467_features)

In [ ]:
df.info()

Step 1: In this step we rounded the decimal points to the nearest 2 decimals to make it easier in reading and interpreting our data

In [ ]:
pd.set_option("display.float_format", "{:.2f}".format)
df.describe()

Step 2 : In this step i started checking the amount of missing (null values)that i have and understand the data type of each attribute to understand my dataset in better way.

In [ ]:
print("\n--- Dataset Info ---")
df.info()

print("\n--- Missing Values (% per column) ---")
missing_percent = df.isnull().mean() * 100
print(missing_percent)

print("\n--- Descriptive Statistics (Numeric Columns) ---")
df.describe()
df.isnull().sum()

Step 3 : In this step i wanted to see the number of unique values in each column to know wether i will need to drop any column or not

In [ ]:
for column in df.columns:
    print(f"{column}: Number of unique values {df[column].nunique()}")
    print("==========================================================")

Step 4 : this step for extra visualization for my dataset and better understanding of the features

In [ ]:
for column in df.columns:
    print(f"{column} : {df[column].unique()}")
    print("====================================")

## **Data Cleaning**

Step 5 : we insured that there is no duplication to ensure data consistency

In [ ]:
df.duplicated().sum()
df = df.drop_duplicates()
df

Step 6 : To address the impossible age values, we replaced outliers (like 5 or 200) with the median age of the valid dataset. This ensures that extreme, unrealistic numbers don't distort the "average" or mislead the machine learning model, while still allowing us to keep the rest of the data in those rows.as thgese numbers may be errors and we did not delete them because we will lose so much insightful data about churn or (To address unrealistic age values (e.g., ages below 16 or above 90), logical boundaries were defined. Ages falling outside this range were considered data entry errors. Instead of removing these records, the invalid age values were replaced with the median age computed from the valid age range. Median imputation was chosen due to its robustness against extreme values, ensuring that unrealistic observations did not distort the age distribution while preserving valuable behavioral data for churn analysis.)

In [ ]:
# 1. Identify the logical bounds
min_age = 16
max_age = 90

# 2. Calculate the Median of the "sane" ages
# We calculate median only from people between 16 and 90
valid_ages = df[(df['Age'] >= min_age) & (df['Age'] <= max_age)]['Age']
median_age = valid_ages.median()

# 3. Replace the "impossible" ages with that median
# This handles the 5-year-olds and the 200-year-olds in one go
df.loc[(df['Age'] < min_age) | (df['Age'] > max_age), 'Age'] = median_age

print(f"Ages outside {min_age}-{max_age} have been replaced with the median: {median_age}")

plotting age distribuion after cleaning

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))

# Plotting the cleaned age
sns.histplot(df['Age'], kde=True, color='green')

plt.title('Distribution of Age after Cleaning')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.3)

plt.show()

Step 7 : In this step we checked the amount of negative purchases in our dataset as it might affect the accuracy of our model we found it very small percentage (0.08% of our dataset) so we replaced them with zero which means that this customer doesnot have history or good history with purcahses. to avoid also overitting

In [ ]:
# Count how many negatives exist before fixing
negative_count = (df['Total_Purchases'] < 0).sum()
total_count = len(df)
percentage = (negative_count / total_count) * 100

print(f"Negative values: {negative_count} ({percentage:.2f}%)")

In [ ]:
# Handle negative Total_Purchases (0.08% of data)
# We set to 0 to represent 'no valid purchase history'
df.loc[df['Total_Purchases'] < 0, 'Total_Purchases'] = 0

Step 8 : Missing values in numerical columns were imputed with their respective means, while missing values in categorical columns were imputed with their modes to ensure a complete dataset for further analysis.and we replaced the nulls and naans in the wishlist with 0 instead of mean as it makes more sense

In [ ]:
import numpy as np

# Step 1: Replace various string representations of nulls with np.nan
# This also handles cases where numerical columns might have these strings, converting them to NaN
df = df.replace(['Nan', 'nan', 'NA', 'N/A', 'na', 'None', 'none', '', ' '], np.nan)

if 'Wishlist' in df.columns:
    df['Wishlist'] = df['Wishlist'].fillna(0)
    print("Column 'Wishlist' nulls have been replaced with 0.")

# Ensure empty strings in object columns are also treated as NaN
for col in df.select_dtypes(include=['object']):
    df[col] = df[col].apply(lambda x: np.nan if isinstance(x, str) and x.strip() == '' else x)


# Step 2: Impute numerical columns with their mean
numerical_cols_with_nulls = df.select_dtypes(include=np.number).columns[df.select_dtypes(include=np.number).isnull().any()].tolist()

print(f"\nNumerical columns to impute with mean: {numerical_cols_with_nulls}")

for col in numerical_cols_with_nulls:
    if col in df.columns:
        mean_val = df[col].mean()
        df[col] = df[col].fillna(mean_val)

# Step 3: Impute categorical columns with their mode
categorical_cols_with_nulls = df.select_dtypes(include='object').columns[df.select_dtypes(include='object').isnull().any()].tolist()

print(f"\nCategorical columns to impute with mode: {categorical_cols_with_nulls}")

for col in categorical_cols_with_nulls:
    if col in df.columns:
        # Ensure mode imputation is robust to all NaNs in a column
        if not df[col].mode().empty:
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)
        else:
            print(f"Warning: Column '{col}' contains only NaN values or no mode could be determined. Skipping mode imputation.")


print("\n--- Missing Values after Imputation ---")
print(df.isnull().sum())

Step 9 : "Categorical features were converted from the object data type to category to optimize memory usage and computational performance. By storing these strings as integer-mapped labels rather than raw objects, we significantly reduced the dataframe's memory footprint and decreased the processing time for downstream grouping and sorting operations."

In [ ]:
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype('category')
df.info()

Step 10: During data validation, 30 observations (0.06% of the dataset) contained invalid cart abandonment rates exceeding 100%, which is mathematically inconsistent for a percentage-based metric. These records were removed to maintain data integrity.

In [ ]:
# Count how many values are greater than 100
count_above_100 = (df['Cart_Abandonment_Rate'] > 100).sum()

print("Number of values above 100:", count_above_100)
df['Cart_Abandonment_Rate'].describe()
df = df[(df['Cart_Abandonment_Rate'] >= 0) &
        (df['Cart_Abandonment_Rate'] <= 100)]

print("New dataset shape:", df.shape)

Step 11 : Data validation identified 207 observations (0.41% of the dataset) with discount usage rates exceeding 100%, which violates the definition of a percentage-based metric. These inconsistent records were removed to preserve analytical accuracy.

In [ ]:
# Count how many values are greater than 100
count_above_100 = (df['Discount_Usage_Rate'] > 100).sum()

print("Number of values above 100:", count_above_100)
# Count negative values
count_negative = (df['Discount_Usage_Rate'] < 0).sum()

print("Number of negative values:", count_negative)
df = df[(df['Discount_Usage_Rate'] >= 0) &
        (df['Discount_Usage_Rate'] <= 100)]

print("New dataset shape:", df.shape)

Step 12 : Both Returns_Rate and Email_Open_Rate columns are statistically right so there is no need to change anything

In [ ]:
# Count how many values are greater than 100
count_above_100 = (df['Returns_Rate'] > 100).sum()

print("Number of values above 100:", count_above_100)
# Count negative values
count_negative = (df['Returns_Rate'] < 0).sum()

print("Number of negative values:", count_negative)

In [ ]:
# Count how many values are greater than 100
count_above_100 = (df['Email_Open_Rate'] > 100).sum()

print("Number of values above 100:", count_above_100)
# Count negative values
count_negative = (df['Email_Open_Rate'] < 0).sum()

print("Number of negative values:", count_negative)

Step 6: handle class imbalance

In [ ]:
print(df['Churned'].value_counts())

## **Feature Engineering**

Step 13 : In this step i transformed the column Membership_Years to Membership_Days to make all calculation standardizede using days

In [ ]:
if 'Membership_Years' in df.columns:
    df['Membership_Days'] = df['Membership_Years'] * 365

    # Optional: Drop the old Years column to keep the data clean
    df.drop(columns=['Membership_Years'], inplace=True)

print("Membership has been harmonized to 'Days' to match purchase recency.")
df

Step 14 : In this step we featured engineered the RFM_score column from the columns Days_Since_Last_Purchase, Total_Purchases, Lifetime_Value to use it in prediction and then we dropped the columns R_score,F_score,M_scor',R, F, M to avoid duplicates

In [ ]:
import pandas as pd

# --- Step 1: Create R, F, M base columns ---
df['R'] = df['Days_Since_Last_Purchase']
df['F'] = df['Total_Purchases']
df['M'] = df['Lifetime_Value']


# --- Step 2: Create RFM Scores using Quantiles (1 to 5) ---

# Recency: lower value = better → reverse scoring
df['R_score'] = pd.qcut(df['R'], 5, labels=[5,4,3,2,1])

# Frequency: higher value = better
df['F_score'] = pd.qcut(df['F'], 5, labels=[1,2,3,4,5])

# Monetary: higher value = better
df['M_score'] = pd.qcut(df['M'], 5, labels=[1,2,3,4,5])


# --- Step 3: Convert scores to integers ---
df['R_score'] = df['R_score'].astype(int)
df['F_score'] = df['F_score'].astype(int)
df['M_score'] = df['M_score'].astype(int)


# --- Step 4: Create Final RFM Score ---
df['RFM_Score'] = (
    df['R_score'] +
    df['F_score'] +
    df['M_score']
)

print("RFM feature engineering completed.")
print(df[['R_score','F_score','M_score','RFM_Score']].head())
df

In [ ]:
df.drop(['R_score','F_score','M_score','R', 'F', 'M'], axis=1, inplace=True, errors='ignore')
df

In [ ]:
df.describe()

Step 15 : The Interquartile Range (IQR) method was applied to detect and remove extreme outliers from numerical features. Observations falling outside the range defined by (Q1-1.5*IQR) and (Q3+1.5 *IQR) were removed. This step helps improve the robustness of machine learning models, particularly those sensitive to extreme values such as Logistic Regression , Support Vector Machines and Naiive Bayes.


In [ ]:
# Select numerical columns
numeric_cols = df.select_dtypes(include=['float64','int64']).columns
numeric_cols = numeric_cols.drop('Churned')  # Exclude target variable

# Calculate IQR
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

# Remove outliers
df = df[~((df[numeric_cols] < (Q1 - 1.5 * IQR)) |
          (df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

print("New dataset shape:", df.shape)

# **Data Preparation for Modelling**

One Hot encoding

The dataset was divided into predictor variables (X) and the target variable (y). The target variable represents customer churn status, while the predictor variables include demographic, behavioral, and transactional features.

In [ ]:
import pandas as pd

# Separate features and target
X = df.drop("Churned", axis=1)
y = df["Churned"]

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include='category').columns

# Apply one-hot encoding to categorical columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

#print("One-hot encoding completed.")

After one-hot encoding, the `X` DataFrame (our feature set for modeling) has its categorical columns converted into numerical representations. Let's inspect `X` to see its structure and data types.

In [ ]:
#print("Shape of X after one-hot encoding:", X.shape)
#print("First 5 rows of X:")
#display(X.head())


Scaling

## **Modelling**

The dataset was split into training and testing sets using an 80/20 ratio. Stratified sampling was applied to preserve the original churn distribution in both subsets, ensuring reliable model evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Feature scaling was applied using StandardScaler to normalize the distribution of numerical variables. This ensures that features with larger magnitudes do not dominate the learning process in algorithms sensitive to feature scale.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**feature selection**

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

rfe = RFE(model, n_features_to_select=15)  # try different numbers
X_rfe = rfe.fit_transform(X, y)

selected_features = X.columns[rfe.support_]
print(selected_features)

Multiple machine learning algorithms were implemented to compare predictive performance, including Logistic Regression, Naïve Bayes, Decision Trees, Support Vector Machines, Random Forest, AdaBoost, and XGBoost.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

models = {

    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Naive Bayes": GaussianNB(),

    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "SVM": SVC(probability=True, random_state=42),

    "Random Forest": RandomForestClassifier(random_state=42),

    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),

    "AdaBoost": AdaBoostClassifier(random_state=42)

}

Five-fold cross-validation was used to evaluate model performance. This technique divides the training data into five subsets, allowing the model to be trained and validated multiple times to reduce overfitting and improve reliability.

In [ ]:
from sklearn.model_selection import cross_val_score

## **Models with class balancing**

SMOTE creates synthetic samples of the minority class (churn = 1) so that the model learns equally from both classes.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("Original training shape:", X_train_scaled.shape)
print("Balanced training shape:", X_train_balanced.shape)

### **1) Logistic regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

log_model = LogisticRegression(max_iter=1000, random_state=42)

scores = cross_val_score(
    log_model,
    X_train_balanced,
    y_train_balanced,
    cv=5,
    scoring="roc_auc"
)
print("Individual Fold Scores:", scores)
log_balance = scores.mean()

print("Logistic Regression ROC-AUC (Balanced):", log_balance)

### **2) Naive Byaes**

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()

scores = cross_val_score(
    nb_model,
    X_train_balanced,
    y_train_balanced,
    cv=5,
    scoring="roc_auc"
)

nb_balance = scores.mean()
print("Individual Fold Scores:", scores)

print("Naive Bayes ROC-AUC (Balanced):", nb_balance)

### **3) Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=10,
    random_state=42
)

scores = cross_val_score(
    dt_model,
    X_train_balanced,
    y_train_balanced,
    cv=5,
    scoring="roc_auc"
)

dt_balance = scores.mean()

print("Decision Tree ROC-AUC (Balanced):", dt_balance)

### **4) SVM**

In [ ]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    C=1,
    max_iter=2000,
    random_state=42
)

scores = cross_val_score(
    svm_model,
    X_train_balanced,
    y_train_balanced,
    cv=5,
    scoring="roc_auc"
)

svm_balance = scores.mean()

print("SVM ROC-AUC (Balanced):", svm_balance)

### **5) Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

scores = cross_val_score(
    rf_model,
    X_train_balanced,
    y_train_balanced,
    cv=5,
    scoring="roc_auc"
)

rf_balance = scores.mean()

print("Random Forest ROC-AUC (Balanced):", rf_balance)

### **6) XGBoost**

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=50,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    eval_metric="logloss"
)

scores = cross_val_score(
    xgb_model,
    X_train_balanced,
    y_train_balanced,
    cv=5,
    scoring="roc_auc"
)

xgb_balance = scores.mean()

print("XGBoost ROC-AUC (Balanced):", xgb_balance)

### **7) AdaBoost**

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(
    n_estimators=50,
    learning_rate=1,
    random_state=42
)

scores = cross_val_score(
    ada_model,
    X_train_balanced,
    y_train_balanced,
    cv=3,
    scoring="roc_auc"
)

ada_balance = scores.mean()

print("AdaBoost ROC-AUC (Balanced):", ada_balance)

## **Evaluation**

The performance of the implemented machine learning models was evaluated using multiple classification metrics including Accuracy, Precision, Recall, F1-score, and ROC-AUC. While accuracy provides an overall measure of correct predictions, precision and recall are particularly important in churn prediction because they measure how effectively the model identifies customers who are likely to leave. The ROC-AUC metric was used as the primary comparison criterion since it evaluates the model's ability to distinguish between churners and non-churners across different decision thresholds.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    return accuracy, precision, recall, f1, roc_auc

# Evaluate each model and store its metrics
log_metrics = evaluate_model(models["Logistic Regression"], X_train_balanced, y_train_balanced, X_test_scaled, y_test)
nb_metrics = evaluate_model(models["Naive Bayes"], X_train_balanced, y_train_balanced, X_test_scaled, y_test)
dt_metrics = evaluate_model(DecisionTreeClassifier(max_depth=10, random_state=42), X_train_balanced, y_train_balanced, X_test_scaled, y_test) # Using the same max_depth as in cross_val_score
svm_metrics = evaluate_model(SVC(probability=True, random_state=42, max_iter=2000), X_train_balanced, y_train_balanced, X_test_scaled, y_test) # Using probability=True for roc_auc and max_iter for convergence
rf_metrics = evaluate_model(RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1), X_train_balanced, y_train_balanced, X_test_scaled, y_test) # Using same params as in cross_val_score
xgb_metrics = evaluate_model(XGBClassifier(n_estimators=50, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42, eval_metric="logloss"), X_train_balanced, y_train_balanced, X_test_scaled, y_test) # Using same params as in cross_val_score
ada_metrics = evaluate_model(AdaBoostClassifier(n_estimators=50, learning_rate=1, random_state=42), X_train_balanced, y_train_balanced, X_test_scaled, y_test) # Using same params as in cross_val_score

import pandas as pd

results_table = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Decision Tree",
        "Random Forest",
        "SVM",
        "XGBoost",
        "AdaBoost"
    ],

    "Accuracy": [
        log_metrics[0],
        nb_metrics[0],
        dt_metrics[0],
        rf_metrics[0],
        svm_metrics[0],
        xgb_metrics[0],
        ada_metrics[0]
    ],

    "Precision": [
        log_metrics[1],
        nb_metrics[1],
        dt_metrics[1],
        rf_metrics[1],
        svm_metrics[1],
        xgb_metrics[1],
        ada_metrics[1]
    ],

    "Recall": [
        log_metrics[2],
        nb_metrics[2],
        dt_metrics[2],
        rf_metrics[2],
        svm_metrics[2],
        xgb_metrics[2],
        ada_metrics[2]
    ],

    "F1 Score": [
        log_metrics[3],
        nb_metrics[3],
        dt_metrics[3],
        rf_metrics[3],
        svm_metrics[3],
        xgb_metrics[3],
        ada_metrics[3]
    ],

    "ROC-AUC": [
        log_metrics[4],
        nb_metrics[4],
        dt_metrics[4],
        rf_metrics[4],
        svm_metrics[4],
        xgb_metrics[4],
        ada_metrics[4]
    ]
})

results_table

## **Extras**

In [ ]:
rf_model.fit(X_train_scaled, y_train)

churn_probabilities = rf_model.predict_proba(X_test_scaled)[:,1]

results_df = X_test.copy()

results_df["Actual_Churn"] = y_test
results_df["Churn_Probability"] = churn_probabilities

#results_df['Churn_Probability'].unique()
results_df

### **Create Churn Risk Categories**

In [ ]:
def churn_risk(prob):

    if prob < 0.30:
        return "Low Risk"

    elif prob < 0.60:
        return "Medium Risk"

    else:
        return "High Risk"

results_df["Risk_Level"] = results_df["Churn_Probability"].apply(churn_risk)

results_df.head()

### **Show Probability Distribution**

In [ ]:
import matplotlib.pyplot as plt

plt.hist(results_df["Churn_Probability"], bins=30)
plt.title("Distribution of Predicted Churn Probability")
plt.xlabel("Churn Probability")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
dashboard = results_df.groupby("Risk_Level").size().reset_index(name="Customers")

dashboard["Percentage"] = (dashboard["Customers"] / dashboard["Customers"].sum()) * 100

dashboard

In [ ]:
import matplotlib.pyplot as plt

plt.bar(dashboard["Risk_Level"], dashboard["Customers"])

plt.title("Customer Churn Risk Segmentation")
plt.xlabel("Risk Level")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
plt.pie(
    dashboard["Customers"],
    labels=dashboard["Risk_Level"],
    autopct="%1.1f%%"
)

plt.title("Distribution of Customer Churn Risk")

plt.show()

In [ ]:
xgb_model.fit(X_train_scaled, y_train)

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

feature_importance.head(10)

In [ ]:
import matplotlib.pyplot as plt

top_features = feature_importance.head(10)

plt.barh(top_features["Feature"], top_features["Importance"])

plt.xlabel("Importance Score")
plt.ylabel("Feature")

plt.title("Top 10 Important Features for Churn Prediction")

plt.gca().invert_yaxis()

plt.show()

In [ ]:
!pip install shap
import shap
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_scaled)
shap.summary_plot(shap_values, X_test_scaled)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar")

In [ ]:
import pandas as pd

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_train.columns
)

In [ ]:
shap.decision_plot(
    explainer.expected_value,
    shap_values[0],
    X_test_scaled_df.iloc[0]
)

A SHAP waterfall plot was used to explain the prediction for an individual customer. The visualization illustrates how different features contribute to the final churn prediction generated by the model. Starting from the baseline prediction, each feature either increases or decreases the churn probability. In this example, variables such as cart abandonment rate, customer service calls, and lifetime value significantly reduced the predicted churn probability. Conversely, the number of days since the last purchase slightly increased the churn risk. Overall, the combined effect of these factors resulted in a lower churn prediction for this custome

In [ ]:
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_test_scaled_df.iloc[0],
        feature_names=X_test_scaled_df.columns
    )
)

In [ ]:
import joblib

joblib.dump(xgb_model, "churn_model.pkl")

In [ ]:
import joblib
joblib.dump(scaler, "scaler.pkl")

In [ ]:
import pandas as pd
import joblib

# Load model and scaler
model = joblib.load("churn_model.pkl")
scaler = joblib.load("scaler.pkl")  # if used

def predict_churn(customer_data):
    # Convert to DataFrame
    df = pd.DataFrame([customer_data])

    # Apply same preprocessing (VERY IMPORTANT)

    # Example: convert Membership_Years → Days
    df["Membership_Days"] = df["Membership_Years"] * 365
    df.drop("Membership_Years", axis=1, inplace=True)

    # Create RFM score (as a sum of raw values, as a placeholder to avoid the KeyError)
    # NOTE: This RFM_score calculation is not fully consistent with the quantile-based RFM_Score
    # used during training. A more robust solution would involve storing and applying the quantile bins.
    df["RFM_Score"] = (
    (1 / (df["Days_Since_Last_Purchase"] + 1)) * 100 +
    df["Total_Purchases"] +
    df["Lifetime_Value"] / 100
)
    # Identify categorical columns for one-hot encoding
    categorical_cols = df.select_dtypes(include='object').columns

    # Apply one-hot encoding to categorical columns
    # This part needs to ensure all categories present during training are also present here, even if not in new_customer
    # For a robust solution, you'd save the columns from X_train and reindex/align new_customer df
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    # Align columns with training data (critical for consistent feature order and presence)
    # This assumes `X_train` columns are available or have been saved
    # For simplicity, we'll try to match X_train columns from the global scope if available
    global X_train # Assuming X_train is available globally from previous execution
    if 'X_train' in globals():
        missing_cols = set(X_train.columns) - set(df.columns)
        for c in missing_cols:
            df[c] = 0
        df = df[X_train.columns] # Ensure same order

    # Apply scaling if used
    df_scaled = scaler.transform(df)

    # Predict probability
    probability = model.predict_proba(df_scaled)[0][1]

    return probability

In [ ]:
new_customer = {
    "Age": 30,
    "Gender": "Male",
    "Country": "Egypt",
    "City": "Cairo",
    "Membership_Years": 2,
    "Login_Frequency": 5,
    "Session_Duration_Avg": 10,
    "Pages_Per_Session": 4,
    "Cart_Abandonment_Rate": 0.3,
    "Wishlist_Items": 2,
    "Total_Purchases": 15,
    "Average_Order_Value": 200,
    "Days_Since_Last_Purchase": 40,
    "Discount_Usage_Rate": 0.5,
    "Returns_Rate": 0.1,
    "Email_Open_Rate": 0.6,
    "Customer_Service_Calls": 1,
    "Product_Reviews_Written": 3,
    "Social_Media_Engagement_Score": 50,
    "Mobile_App_Usage": 10,
    "Payment_Method_Diversity": 2,
    "Lifetime_Value": 3000,
    "Credit_Balance": 500,
    "Signup_Quarter": "Q2"
}

In [ ]:
prob = predict_churn(new_customer)
print(f"Churn Probability: {prob:.2f}")

if prob > 0.7:
    print("⚠️ High risk customer – immediate retention action needed")
elif prob > 0.4:
    print("⚠️ Medium risk customer – monitor and engage")
else:
    print("✅ Low risk customer")

In [ ]:

# Extracted from the first row of your table
first_customer = {
    "Age": 43.0,
    "Gender": "Male",
    "Country": "France",
    "City": "Marseille",
    "Membership_Years": 2.9,
    "Login_Frequency": 14.0,
    "Session_Duration_Avg": 27.4,
    "Pages_Per_Session": 6.0,
    "Cart_Abandonment_Rate": 0.63, # Note: Values like 50.6 in tables are often represented as decimals 0.506 or 0.63 in models
    "Wishlist_Items": 3.0,
    "Total_Purchases": 16.0, # Approximate based on layout
    "Average_Order_Value": 59.5, # Placeholder if not explicitly in your snippet
    "Days_Since_Last_Purchase": 20.0, # Placeholder
    "Discount_Usage_Rate": 0.5,
    "Returns_Rate": 0.1,
    "Email_Open_Rate": 17.9,
    "Customer_Service_Calls": 9.0,
    "Product_Reviews_Written": 4.0,
    "Social_Media_Engagement_Score": 16.3,
    "Mobile_App_Usage": 20.8,
    "Payment_Method_Diversity": 1.0,
    "Lifetime_Value": 953.33,
    "Credit_Balance": 2278.00,
    "Signup_Quarter": "Q1"
}

prob = predict_churn(first_customer)
print(f"Churn Probability for Customer 1: {prob:.2f}")

if prob > 0.7:
    print("⚠️ High risk customer – immediate retention action needed")
elif prob > 0.4:
    print("⚠️ Medium risk customer – monitor and engage")
else:
    print("✅ Low risk customer")

In [ ]:
phoenix_customer = {
    "Age": 55.0,
    "Gender": "Male",
    "Country": "USA",
    "City": "Phoenix",
    "Login_Frequency": 26.0,
    "Session_Duration_Avg": 34.9,
    "Pages_Per_Session": 10.8,
    "Cart_Abandonment_Rate": 48.1,
    "Wishlist_Items": 12.0,
    "Total_Purchases": 20.8,
    "Average_Order_Value": 77.19,
    "Days_Since_Last_Purchase": 60.0,
    "Discount_Usage_Rate": 34.08,
    "Returns_Rate": 8.6,
    "Email_Open_Rate": 25.2,
    "Customer_Service_Calls": 1.0,
    "Product_Reviews_Written": 5.0,
    "Social_Media_Engagement_Score": 36.9,
    "Mobile_App_Usage": 30.6,
    "Payment_Method_Diversity": 4.0,
    "Lifetime_Value": 1417.23,
    "Credit_Balance": 2046.0,
    "Signup_Quarter": "Q4",
    "Membership_Days": 839.5,
    "RFM_Score": 9
}

prob = predict_churn(phoenix_customer)

print(f"Customer Location: {phoenix_customer['City']}")
print(f"Calculated Churn Probability: {prob:.2f}")

# Risk Level Output
if prob > 0.60:
    print("Status: High Risk")
elif prob >= 0.30:
    print("Status: Medium Risk")
else:
    print("Status: Low Risk")

In [ ]:
from sklearn.metrics import roc_auc_score

# 'y_test' are the real answers, 'churn_probabilities' are the model's percentages
score = roc_auc_score(y_test, churn_probabilities)
print(f"Model Discrimination Score: {score:.2f}")

In [ ]:
bangalore_customer = {
    "Age": 32.0,
    "Gender": "Female",
    "Country": "India",
    "City": "Bangalore",
    "Login_Frequency": 3.0,
    "Session_Duration_Avg": 27.66,
    "Pages_Per_Session": 4.3,
    "Cart_Abandonment_Rate": 63.4,
    "Wishlist_Items": 0.0,
    "Total_Purchases": 6.5,
    "Average_Order_Value": 168.26,
    "Days_Since_Last_Purchase": 53.0,
    "Discount_Usage_Rate": 6.48,
    "Returns_Rate": 4.4,
    "Email_Open_Rate": 2.0,
    "Customer_Service_Calls": 10.0,
    "Product_Reviews_Written": 0.0,
    "Social_Media_Engagement_Score": 0.0,
    "Mobile_App_Usage": 3.0,
    "Payment_Method_Diversity": 2.0,
    "Lifetime_Value": 960.82,
    "Credit_Balance": 0.0,  # Huge red flag
    "Signup_Quarter": "Q4",
    "Membership_Days": 328.5,
    "Membership_Years": 328.5 / 365
}

# Run the prediction
prob_bangalore = predict_churn(bangalore_customer)

print(f"--- Prediction for Customer in {bangalore_customer['City']} ---")
print(f"Churn Probability: {prob_bangalore:.2f}")

if prob_bangalore > 0.60:
    print("Status: High Risk")
elif prob_bangalore >= 0.30:
    print("Status: Medium Risk")
else:
    print("Status: Low Risk")